# Step 6. 지표 계산

**목표**: 할인 이벤트별 반응률 지표 계산  
**입력**: `discount_history.csv`, `review_daily.csv`, `game_metadata.csv`  
**출력**: `data/analysis_df.csv`

### 핵심 지표
- **할인 반응률** = (할인기간 일평균 리뷰 − 직전30일 일평균) / 직전30일 일평균
- **유지 반응률** = (종료후14일 일평균 − 직전30일 일평균) / 직전30일 일평균

### 통제 변수
- `is_seasonal_sale`: Steam 대형 시즌 세일 기간 포함 여부
- `days_since_release`: 출시 후 경과 일수 (할인 시작일 기준)
- `total_reviews`: 전체 리뷰 수 (인기도 대리 지표)
- `is_multiplayer`: 멀티플레이 여부
- `price_usd`: 정가

In [1]:
import pandas as pd
import numpy as np
from datetime import date

disc = pd.read_csv("../data/discount_history.csv")
rev  = pd.read_csv("../data/review_daily.csv")
meta = pd.read_csv("../data/game_metadata.csv")

disc["discount_start"] = pd.to_datetime(disc["discount_start"])
disc["discount_end"]   = pd.to_datetime(disc["discount_end"])
rev["date"]            = pd.to_datetime(rev["date"])
meta["release_date"]   = pd.to_datetime(meta["release_date"])

print(f"할인 이벤트: {len(disc)}개 / 게임: {disc['appid'].nunique()}개")
print(f"리뷰 일별 데이터: {len(rev):,}행 / 게임: {rev['appid'].nunique()}개")
print(f"메타데이터: {len(meta)}개")

ValueError: time data "Jul 19, 2022" doesn't match format "%Y-%m-%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

## 헬퍼 함수

In [ ]:
# Steam 주요 시즌 세일 기간 (2024~2026)
SEASONAL_SALES = [
    ("2024-03-14", "2024-03-21"),  # Spring Sale 2024
    ("2024-06-27", "2024-07-11"),  # Summer Sale 2024
    ("2024-11-26", "2024-12-03"),  # Autumn Sale 2024
    ("2024-12-19", "2025-01-02"),  # Winter Sale 2024-25
    ("2025-01-23", "2025-02-05"),  # Lunar New Year 2025
    ("2025-03-13", "2025-03-17"),  # Spring Sale 2025
    ("2025-06-26", "2025-07-10"),  # Summer Sale 2025
    ("2025-11-26", "2025-12-02"),  # Autumn Sale 2025
    ("2025-12-18", "2026-01-02"),  # Winter Sale 2025-26
]
SEASONAL_SALES = [(pd.Timestamp(s), pd.Timestamp(e)) for s, e in SEASONAL_SALES]


def is_seasonal(start, end):
    """이벤트 기간이 시즌 세일과 겹치는지 여부"""
    for sale_s, sale_e in SEASONAL_SALES:
        if start <= sale_e and end >= sale_s:
            return True
    return False


def window_avg(daily_series, start, end):
    """
    daily_series: dict-like {date: count} (appid 기준으로 필터된 Series)
    start ~ end 구간의 일평균 리뷰 수 반환.
    구간 내 데이터가 없으면 NaN.
    """
    mask = (daily_series.index >= start) & (daily_series.index < end)
    sub = daily_series[mask]
    if len(sub) == 0:
        return np.nan
    # 기간 일수로 나눔 (데이터 없는 날 = 0으로 처리)
    n_days = (end - start).days
    return sub.sum() / n_days


print("함수 정의 완료")

## 지표 계산

In [ ]:
# 게임별 리뷰 시계열 딕셔너리로 변환 (빠른 조회)
rev_by_game = {}
for appid, grp in rev.groupby("appid"):
    s = grp.set_index("date")["daily_reviews"]
    rev_by_game[appid] = s

PRE_DAYS  = 30
POST_DAYS = 14
MIN_PRE_DAYS  = 14   # 직전 데이터 최소 일수
MIN_POST_DAYS = 7    # 종료 후 데이터 최소 일수

rows = []
skipped = 0

for _, ev in disc.iterrows():
    appid = int(ev["appid"])
    if appid not in rev_by_game:
        skipped += 1
        continue

    daily = rev_by_game[appid]
    start = ev["discount_start"]
    end   = ev["discount_end"]

    pre_start  = start - pd.Timedelta(days=PRE_DAYS)
    post_end   = end   + pd.Timedelta(days=POST_DAYS)

    # 데이터 커버리지 확인
    rev_from = daily.index.min()
    rev_to   = daily.index.max()
    pre_avail  = (start - rev_from).days
    post_avail = (rev_to - end).days

    if pre_avail < MIN_PRE_DAYS or post_avail < MIN_POST_DAYS:
        skipped += 1
        continue

    # 각 구간 일평균
    before_avg = window_avg(daily, pre_start, start)
    during_avg = window_avg(daily, start, end)
    after_avg  = window_avg(daily, end, post_end)

    if pd.isna(before_avg) or before_avg == 0:
        skipped += 1
        continue

    # 핵심 지표
    reaction_rate  = (during_avg - before_avg) / before_avg if not pd.isna(during_avg) else np.nan
    sustained_rate = (after_avg  - before_avg) / before_avg if not pd.isna(after_avg)  else np.nan

    rows.append({
        "appid":          appid,
        "name":           ev["name"],
        "genre_category": ev["genre_category"],
        "discount_start": start,
        "discount_end":   end,
        "discount_pct":   ev["discount_pct"],
        "duration_days":  ev["duration_days"],
        "before_daily_avg": round(before_avg, 2),
        "during_daily_avg": round(during_avg, 2) if not pd.isna(during_avg) else np.nan,
        "after_daily_avg":  round(after_avg,  2) if not pd.isna(after_avg)  else np.nan,
        "reaction_rate":    round(reaction_rate,  4) if not pd.isna(reaction_rate)  else np.nan,
        "sustained_rate":   round(sustained_rate, 4) if not pd.isna(sustained_rate) else np.nan,
        "is_seasonal_sale": is_seasonal(start, end),
    })

result_df = pd.DataFrame(rows)
print(f"계산 완료: {len(result_df)}개 이벤트 (제외: {skipped}개)")

## 통제 변수 병합

In [ ]:
ctrl = meta[["appid", "release_date", "total_reviews", "price_usd", "is_multiplayer"]].copy()
result_df = result_df.merge(ctrl, on="appid", how="left")

# 출시 후 경과 일수 (할인 시작일 기준)
result_df["days_since_release"] = (
    result_df["discount_start"] - result_df["release_date"]
).dt.days

result_df = result_df.drop(columns=["release_date"])

print(f"최종 컬럼: {list(result_df.columns)}")
print(f"Shape: {result_df.shape}")

## 결과 확인

In [ ]:
print("=== 장르별 이벤트 수 ===")
print(result_df.groupby("genre_category")["appid"].count().rename("이벤트 수"))

print("\n=== 장르별 평균 지표 ===")
summary = result_df.groupby("genre_category")[["reaction_rate", "sustained_rate", "discount_pct"]].mean().round(3)
summary.columns = ["할인반응률", "유지반응률", "평균할인율"]
print(summary)

print("\n=== 전체 요약 ===")
print(f"할인 반응률 범위: {result_df['reaction_rate'].min():.2f} ~ {result_df['reaction_rate'].max():.2f}")
print(f"유지 반응률 범위: {result_df['sustained_rate'].min():.2f} ~ {result_df['sustained_rate'].max():.2f}")
print(f"시즌 세일 포함 이벤트: {result_df['is_seasonal_sale'].sum()}개")
print(f"NaN 포함 행:")
print(result_df[["reaction_rate", "sustained_rate"]].isna().sum())

result_df.head(10)

## CSV 저장

In [ ]:
output_path = "../data/analysis_df.csv"
result_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path}")
print(f"Shape: {result_df.shape}")